# Create the Groq Model

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Create a Tool

In [ ]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""

    return a * b

# Bind the Tool to the Model

In [ ]:
model_with_tools = model.bind_tools(
    [multiply]
)

# Ask a Question

In [ ]:
response = model_with_tools.invoke(
    "What is 25 multiplied by 4?"
)

print(response)

In [ ]:
print(response.tool_calls)

# Execute the Tool

In [ ]:
tool_call = response.tool_calls[0]

result = multiply.invoke(
    tool_call["args"]
)

print(result)

# Send Tool Result Back to the Model

In [ ]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"]
)

In [ ]:
final_response = model_with_tools.invoke(
    [
        {
            "role": "user",
            "content": "What is 25 multiplied by 4?"
        },
        response,
        tool_message
    ]
)

print(final_response.content)

# Complete Manual Tool-Calling Example

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage


# --------------------------------
# 1. Model
# --------------------------------

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


# --------------------------------
# 2. Tool
# --------------------------------

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""

    return a * b


# --------------------------------
# 3. Bind Tool
# --------------------------------

model_with_tools = model.bind_tools(
    [multiply]
)


# --------------------------------
# 4. User Question
# --------------------------------

user_message = {
    "role": "user",
    "content": "What is 25 multiplied by 4?"
}


# --------------------------------
# 5. First LLM Call
# --------------------------------

response = model_with_tools.invoke(
    [user_message]
)

print("Tool Calls:")
print(response.tool_calls)


# --------------------------------
# 6. Execute Tool
# --------------------------------

tool_call = response.tool_calls[0]

tool_result = multiply.invoke(
    tool_call["args"]
)

print("Tool Result:")
print(tool_result)


# --------------------------------
# 7. Create ToolMessage
# --------------------------------

tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_call["id"]
)


# --------------------------------
# 8. Second LLM Call
# --------------------------------

final_response = model_with_tools.invoke(
    [
        user_message,
        response,
        tool_message
    ]
)


# --------------------------------
# 9. Final Answer
# --------------------------------

print("Final Answer:")
print(final_response.content)